In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from diagnostic_reasoning import (
    OVRProjectionResult, ovr_project_posterior, project_ovr_to_coherent,
    _project_ovr_to_coherent, common_reference_posterior, normalize_priors,
    validate_posterior, float_vector, representable_exp, require_candidate,
    CERTAINTY_ESTIMATES, qualitative_score,
)

import matplotlib.pyplot as plt
import scienceplots
import numpy as np
from ipywidgets import interact, widgets, VBox, HBox, Layout, Output

plt.style.use('science')
plt.rcParams['text.usetex'] = False

certainty_estimates = {
    "HIGHLY_UNLIKELY": 0.119,
    "UNLIKELY": 0.269,
    "UNCERTAIN": 0.5,
    "LIKELY": 0.731,
    "HIGHLY_LIKELY": 0.881
}

def softmax_regression_predict_multi_common(pre_test_probabilities, weights_of_evidence):
    """Common-reference scores, not raw one-vs-rest LRs."""
    return common_reference_posterior(pre_test_probabilities, weights_of_evidence)


def stacked_chart_pre_post_multi_common(pre_test_probabilities, post_testing_probabilities, dx_names, weights_of_evidence, figsize=(6, 6)):
    """Plot validated probabilities; never normalize away an invalid result."""
    pre = validate_posterior(pre_test_probabilities, dx_names)
    post = validate_posterior(post_testing_probabilities, dx_names)
    scores = float_vector(weights_of_evidence, 'input log scores')
    if scores.shape != pre.shape:
        raise ValueError('input log scores and probabilities must match.')
    fitted = None
    if fitted is not None and float_vector(fitted, 'fitted OVR LRs').shape != pre.shape:
        raise ValueError('fitted OVR LRs and probabilities must match.')
    mode = "common-reference"
    fig, ax = plt.subplots(figsize=(7.3, max(4.5, 3.4 + .22 * len(pre))))
    bottom = np.zeros(2)
    for i, name in enumerate(dx_names):
        values = np.array([pre[i], post[i]])
        label = f'{name} | input {mode} LR {np.exp(scores[i]):.3g}'
        if fitted is not None:
            label += f' | fitted OVR LR {fitted[i]:.3g}'
        bars = ax.bar([0, 1], values, bottom=bottom, width=.32, label=label)
        for bar, value in zip(bars, values):
            if value >= .04:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+value/2,
                        f'{value:.3f}', ha='center', va='center', color='white', fontsize=10)
        bottom += values
        ax.plot([0, 1], bottom, '--', color='gray', lw=.8)
    ax.set(xticks=[0, 1], xticklabels=['Before', 'After'], ylabel='Probability', ylim=(0, 1),
           title=('Coherent fit candidate (minimum unverified)' if mode == 'OVR' else 'Common-reference Bayes update'))
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.12), fontsize=8.5)
    fig.tight_layout()
    plt.show()


# Generalized wrapper function for n diagnoses
def generalized_predictor_wrapper_multi_common(diagnoses):
    """Relative prior weights and synthetic count aggregates."""
    names = [d['name'] for d in diagnoses]
    raw = np.asarray([CERTAINTY_ESTIMATES[d['pretest']] for d in diagnoses], dtype=float)
    scores = float_vector([qualitative_score(d['major_positive'], d['major_negative'], d['minor_positive'], d['minor_negative']) for d in diagnoses], 'scores')
    mode = 'softmax_scores'
    if mode not in ('ovr_projection', 'softmax_scores'):
        raise ValueError('Unknown evidence mode.')
    priors = normalize_priors(raw, ovr=mode == 'ovr_projection')
    if len(scores) != len(priors):
        raise ValueError('scores and priors must have matching lengths.')
    fitted, fitted_log = None, None
    if mode == 'ovr_projection':
        result = require_candidate(ovr_project_posterior(priors, representable_exp(scores, 'input OVR LRs'), reg=0.))
        priors, post = result.diagnostics['priors'], result.posterior
        fitted, fitted_log = result.fitted_ovr_lr, result.diagnostics['fitted_log_ovr_lr']
        print('Input: synthetic aggregate interpreted as OVR; one projection, not a joint-evidence model.')
    else:
        post = common_reference_posterior(priors, scores)
        print('Common-reference class log-likelihood scores. Counts are a synthetic teaching heuristic.')
    validate_posterior(post, names)
    print(f"{'Diagnosis':<20}{'Raw weight':<12}{'Prior':<12}{'Input log score':<18}{'Posterior':<12}")
    for i, name in enumerate(names):
        print(f"{name:<20}{raw[i]:<12.4g}{priors[i]:<12.4g}{scores[i]:<18.4g}{post[i]:<12.4g}")
    if fitted is not None:
        print('Input OVR LR | Fitted OVR LR | Fitted log OVR LR')
        for i, name in enumerate(names):
            print(f"{name}: {np.exp(scores[i]):.6g} | {fitted[i]:.6g} | {fitted_log[i]:.6g}")
    return dict(names=names, pre_test_probs=raw, norm_pre_test_probs=priors,
                weights_of_evidence=scores, predicted_probs=post,
                fitted_ovr_lr=fitted, fitted_loglr=fitted_log)


# Dynamic widget for n diagnoses
def create_diagnosis_widget_multi_common():
    """
    Creates an interactive widget for entering diagnosis details and calculating post-test probabilities
    using softmax regression, displaying the stacked chart dynamically.
    
    Returns:
        VBox: A vertical box containing the interactive widget.
    """
    # Define dynamic widgets for n diagnoses
    diagnosis_count = widgets.IntSlider(value=1, min=1, max=10, step=1, description="Diagnoses")
    container = VBox()
    chart_output = Output()  # Output widget to display the chart

    # Function to update the widgets based on the number of diagnoses
    def update_widgets(change):
        if change['name'] == 'value':  # Ensure the change is triggered by the slider
            container.children = []  # Clear the container before updating
            for i in range(change['new']):  # Iterate over the new diagnosis count
                name_widget = widgets.Text(value=f"Diagnosis {i+1}", description="Name:")
                pretest_widget = widgets.Dropdown(
                    options=['HIGHLY_UNLIKELY', 'UNLIKELY', 'UNCERTAIN', 'LIKELY', 'HIGHLY_LIKELY'],
                    value='UNCERTAIN',
                    description="Prior weight:"
                )
                major_pos_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Major+")
                major_neg_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Major-")
                minor_pos_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Minor+")
                minor_neg_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Minor-")

                # Combine all widgets for one diagnosis
                diagnosis_box = VBox([name_widget, pretest_widget, major_pos_widget, major_neg_widget, minor_pos_widget, minor_neg_widget])
                container.children += (diagnosis_box,)  # Add to the container

    # Observe changes in the slider and update widgets accordingly
    diagnosis_count.observe(update_widgets, names='value')

    # Trigger an initial update
    update_widgets({'name': 'value', 'new': diagnosis_count.value})

    # Function to gather inputs from the widgets
    def gather_inputs():
        diagnoses = []
        for diagnosis_box in container.children:
            name = diagnosis_box.children[0].value
            pretest = diagnosis_box.children[1].value
            major_positive = diagnosis_box.children[2].value
            major_negative = diagnosis_box.children[3].value
            minor_positive = diagnosis_box.children[4].value
            minor_negative = diagnosis_box.children[5].value
            diagnoses.append({
                'name': name,
                'pretest': pretest,
                'major_positive': major_positive,
                'major_negative': major_negative,
                'minor_positive': minor_positive,
                'minor_negative': minor_negative
            })

        # Clear chart output before displaying a new one
        chart_output.clear_output()

        # Compute results and display chart
        with chart_output:
            try:
                results = generalized_predictor_wrapper_multi_common(diagnoses)
                stacked_chart_pre_post_multi_common(
                    results['norm_pre_test_probs'],
                    results['predicted_probs'],
                    results['names'],
                    results['weights_of_evidence']
                )
            except (ValueError, FloatingPointError, OverflowError) as exc:
                print(f"Cannot calculate: {exc}")

    # Add a button to trigger the calculation
    calculate_button = widgets.Button(description="Calculate")
    calculate_button.on_click(lambda x: gather_inputs())

    # Combine the slider, container, and calculate button
    return HBox([VBox([diagnosis_count, container, calculate_button]), chart_output])


widget_multi_common = create_diagnosis_widget_multi_common()
display(widget_multi_common)
# Compatibility aliases; retained callbacks use the distinct names above.
softmax_regression_predict = softmax_regression_predict_multi_common
stacked_chart_pre_post = stacked_chart_pre_post_multi_common
generalized_predictor_wrapper = generalized_predictor_wrapper_multi_common
create_diagnosis_widget = create_diagnosis_widget_multi_common


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from diagnostic_reasoning import (
    OVRProjectionResult, ovr_project_posterior, project_ovr_to_coherent,
    _project_ovr_to_coherent, common_reference_posterior, normalize_priors,
    validate_posterior, float_vector, representable_exp, require_candidate,
    CERTAINTY_ESTIMATES, qualitative_score,
)

import matplotlib.pyplot as plt
import scienceplots
import numpy as np
from ipywidgets import interact, widgets, VBox, HBox, Layout, Output

plt.style.use('science')
plt.rcParams['text.usetex'] = False

# --------------------------------------------------------------------
# Qualitative pretest → numeric prior (edit these if you recalibrate)
# --------------------------------------------------------------------
certainty_estimates = {
    "HIGHLY_UNLIKELY": 0.119,
    "UNLIKELY":        0.269,
    "UNCERTAIN":       0.500,
    "LIKELY":          0.731,
    "HIGHLY_LIKELY":   0.881
}

# ====================================================================
# Bayes‑coherent OVR projection (mixture‑aware coupling)
#   Inputs:
#     priors π_k  (sum to 1)
#     ovr_lr λ_k  (LR for k vs "not k")  -- here λ_k = exp(weight_of_evidence_k)
#   Output:
#     coherent posterior P_k ∝ π_k * exp(s_k)
#     fitted OVR LRs implied by the model
# ====================================================================





# -------------------------------------------------------------
# Visualization: show Input vs Fitted OVR LR, pre vs post bars
# -------------------------------------------------------------
def _stacked_chart_pre_post_multi_ovr(pre_test_probabilities, post_testing_probabilities, dx_names, input_loglr, fitted_ovr_lr=None, figsize=(6, 6)):
    """Plot validated probabilities; never normalize away an invalid result."""
    pre = validate_posterior(pre_test_probabilities, dx_names)
    post = validate_posterior(post_testing_probabilities, dx_names)
    scores = float_vector(input_loglr, 'input log scores')
    if scores.shape != pre.shape:
        raise ValueError('input log scores and probabilities must match.')
    fitted = fitted_ovr_lr
    if fitted is not None and float_vector(fitted, 'fitted OVR LRs').shape != pre.shape:
        raise ValueError('fitted OVR LRs and probabilities must match.')
    mode = "OVR"
    fig, ax = plt.subplots(figsize=(7.3, max(4.5, 3.4 + .22 * len(pre))))
    bottom = np.zeros(2)
    for i, name in enumerate(dx_names):
        values = np.array([pre[i], post[i]])
        label = f'{name} | input {mode} LR {np.exp(scores[i]):.3g}'
        if fitted is not None:
            label += f' | fitted OVR LR {fitted[i]:.3g}'
        bars = ax.bar([0, 1], values, bottom=bottom, width=.32, label=label)
        for bar, value in zip(bars, values):
            if value >= .04:
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_y()+value/2,
                        f'{value:.3f}', ha='center', va='center', color='white', fontsize=10)
        bottom += values
        ax.plot([0, 1], bottom, '--', color='gray', lw=.8)
    ax.set(xticks=[0, 1], xticklabels=['Before', 'After'], ylabel='Probability', ylim=(0, 1),
           title=('Coherent fit candidate (minimum unverified)' if mode == 'OVR' else 'Common-reference Bayes update'))
    ax.legend(loc='upper center', bbox_to_anchor=(.5, -.12), fontsize=8.5)
    fig.tight_layout()
    plt.show()

# ====================================================================
# Your app‑specific mapping from qualitative finding counts to weights
#   Here we interpret 'weight_of_evidence' as log OVR LR for class k.
#   Edit the coefficients if you recalibrate those qualitative bins.
# ====================================================================
# Compatibility name for the canonical synthetic count aggregate.
_compute_weight_of_evidence = qualitative_score

# ====================================================================
# Wrapper (OVR projection): assembles inputs, runs solver, prints table
# ====================================================================
def generalized_predictor_wrapper_multi_ovr(diagnoses, reg=1e-6):
    """Relative prior weights and synthetic count aggregates."""
    names = [d['name'] for d in diagnoses]
    raw = np.asarray([CERTAINTY_ESTIMATES[d['pretest']] for d in diagnoses], dtype=float)
    scores = float_vector([qualitative_score(d['major_positive'], d['major_negative'], d['minor_positive'], d['minor_negative']) for d in diagnoses], 'scores')
    mode = 'ovr_projection'
    if mode not in ('ovr_projection', 'softmax_scores'):
        raise ValueError('Unknown evidence mode.')
    priors = normalize_priors(raw, ovr=mode == 'ovr_projection')
    if len(scores) != len(priors):
        raise ValueError('scores and priors must have matching lengths.')
    fitted, fitted_log = None, None
    if mode == 'ovr_projection':
        result = require_candidate(ovr_project_posterior(priors, representable_exp(scores, 'input OVR LRs'), reg=reg))
        priors, post = result.diagnostics['priors'], result.posterior
        fitted, fitted_log = result.fitted_ovr_lr, result.diagnostics['fitted_log_ovr_lr']
        print('Input: synthetic aggregate interpreted as OVR; one projection, not a joint-evidence model.')
    else:
        post = common_reference_posterior(priors, scores)
        print('Common-reference class log-likelihood scores. Counts are a synthetic teaching heuristic.')
    validate_posterior(post, names)
    print(f"{'Diagnosis':<20}{'Raw weight':<12}{'Prior':<12}{'Input log score':<18}{'Posterior':<12}")
    for i, name in enumerate(names):
        print(f"{name:<20}{raw[i]:<12.4g}{priors[i]:<12.4g}{scores[i]:<18.4g}{post[i]:<12.4g}")
    if fitted is not None:
        print('Input OVR LR | Fitted OVR LR | Fitted log OVR LR')
        for i, name in enumerate(names):
            print(f"{name}: {np.exp(scores[i]):.6g} | {fitted[i]:.6g} | {fitted_log[i]:.6g}")
    return dict(names=names, pre_test_probs=raw, norm_pre_test_probs=priors,
                weights_of_evidence=scores, predicted_probs=post,
                fitted_ovr_lr=fitted, fitted_loglr=fitted_log)

# ====================================================================
# Interactive widget
# ====================================================================
def create_diagnosis_widget_multi_ovr():
    # OVR needs at least 2 classes
    diagnosis_count = widgets.IntSlider(value=3, min=2, max=12, step=1, description="Diagnoses")
    reg_slider = widgets.FloatLogSlider(value=1e-6, base=10, min=-12, max=-2, step=0.5,
                                        description="Reg (L2)", readout_format=".1e")
    zero_ridge = widgets.Checkbox(value=False, description="No ridge (reg=0)")

    container = VBox()
    chart_output = Output()

    def update_widgets(change):
        if change['name'] == 'value':
            children = []
            for i in range(change['new']):
                name_widget = widgets.Text(value=f"Diagnosis {i+1}", description="Name:")
                pretest_widget = widgets.Dropdown(
                    options=['HIGHLY_UNLIKELY', 'UNLIKELY', 'UNCERTAIN', 'LIKELY', 'HIGHLY_LIKELY'],
                    value='UNCERTAIN', description="Prior weight:"
                )
                major_pos_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Major+")
                major_neg_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Major-")
                minor_pos_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Minor+")
                minor_neg_widget = widgets.IntSlider(value=0, min=0, max=10, step=1, description="Minor-")
                children.append(VBox([name_widget, pretest_widget,
                                      major_pos_widget, major_neg_widget,
                                      minor_pos_widget, minor_neg_widget]))
            container.children = tuple(children)

    diagnosis_count.observe(update_widgets, names='value')
    update_widgets({'name': 'value', 'new': diagnosis_count.value})

    def gather_inputs(_=None):
        diagnoses = []
        for box in container.children:
            diagnoses.append({
                'name':            box.children[0].value,
                'pretest':         box.children[1].value,
                'major_positive':  box.children[2].value,
                'major_negative':  box.children[3].value,
                'minor_positive':  box.children[4].value,
                'minor_negative':  box.children[5].value
            })

        chart_output.clear_output()
        with chart_output:
            try:
                results = generalized_predictor_wrapper_multi_ovr(diagnoses, reg=(0. if zero_ridge.value else float(reg_slider.value)))
                _stacked_chart_pre_post_multi_ovr(
                    results['norm_pre_test_probs'],
                    results['predicted_probs'],
                    results['names'],
                    results['weights_of_evidence'],     # input log OVR LR (for legend)
                    fitted_ovr_lr=results['fitted_ovr_lr']
                )
            except (ValueError, FloatingPointError, OverflowError) as exc:
                print(f"Cannot calculate: {exc}")

    calculate_button = widgets.Button(description="Calculate")
    calculate_button.on_click(gather_inputs)

    left = VBox([diagnosis_count, VBox([reg_slider, zero_ridge]), container, calculate_button])
    return HBox([left, chart_output])


widget_multi_ovr = create_diagnosis_widget_multi_ovr()
display(widget_multi_ovr)
# Compatibility aliases; retained callbacks use the distinct names above.
_stacked_chart_pre_post = _stacked_chart_pre_post_multi_ovr
generalized_predictor_wrapper = generalized_predictor_wrapper_multi_ovr
create_diagnosis_widget = create_diagnosis_widget_multi_ovr
